## Menghubungkan Google Drive dan Pengaturan Folder Output

In [1]:


import os
import re
import pandas as pd
import json
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

BASE_DRIVE_DIR = '/content/drive/MyDrive/CBR/'


RAW_DIR = os.path.join(BASE_DRIVE_DIR, 'data/raw/')


PROCESSED_DIR = os.path.join(BASE_DRIVE_DIR, 'data/processed/')
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Direktori: \n- Input Data: {RAW_DIR}\n- Output Hasil: {PROCESSED_DIR}")

Mounted at /content/drive
Direktori: 
- Input Data: /content/drive/MyDrive/CBR/data/raw/
- Output Hasil: /content/drive/MyDrive/CBR/data/processed/


## Ekstraksi Fitur




In [2]:


def extract_case_features(text):
    """
    Mengekstrak metadata dan konten kunci putusan wanprestasi dengan sistem fallback berlapis.
    """
    if not text or len(text.strip()) < 10:
        text = "Perkara gugatan wanprestasi atas ingkar janji pemenuhan perjanjian kemitraan atau utang piutang."

    # Ekstraksi Nomor Perkara
    no_perkara_match = re.search(r'nomor\s+(\d+/[^\s/]+/20\d{2}/[^\s]+)', text)
    no_perkara = no_perkara_match.group(1).upper() if no_perkara_match else "Tidak Terdeteksi"

    # Ekstraksi Tanggal Putusan
    tanggal_match = re.search(r'(?:tanggal|putusan\s+diucapkan)\s+(\d{1,2}\s+[a-zA-Z]+\s+20\d{2})', text)
    tanggal = tanggal_match.group(1) if tanggal_match else "Tidak Terdeteksi"

    # Ekstraksi Nama Pihak (Sistem Fleksibel)
    penggugat = "Tidak Terdeteksi"
    tergugat = "Tidak Terdeteksi"

    p1_p = re.search(r'(?:nama|selanjutnya\s+disebut\s+sebagai)\s+([^,.\n]+)\s+.*sebagai\s+penggugat', text, re.IGNORECASE)
    p1_t = re.search(r'(?:nama|selanjutnya\s+disebut\s+sebagai)\s+([^,.\n]+)\s+.*sebagai\s+tergugat', text, re.IGNORECASE)

    if p1_p: penggugat = p1_p.group(1).strip().upper()
    if p1_t: tergugat = p1_t.group(1).strip().upper()

    if penggugat == "Tidak Terdeteksi" or tergugat == "Tidak Terdeteksi":
        pihak_alt = re.search(r'antara\s*:\s*([^,.\n]+)\s+.*?melawan\s*:\s*([^,.\n]+)', text, re.IGNORECASE)
        if pihak_alt:
            if penggugat == "Tidak Terdeteksi": penggugat = pihak_alt.group(1).strip().upper()
            if tergugat == "Tidak Terdeteksi": tergugat = pihak_alt.group(2).strip().upper()

    # Pembersihan noise pada nama pihak
    penggugat = re.sub(r'^(bahwa|melawan|berdasarkan|mengadili)\s+', '', penggugat, flags=re.IGNORECASE).strip()
    tergugat = re.sub(r'^(bahwa|melawan|berdasarkan|mengadili)\s+', '', tergugat, flags=re.IGNORECASE).strip()
    pihak = f"{penggugat} VS {tergugat}"

    # Ekstraksi Ringkasan Fakta (MENGGUNAKAN MULTI-ANCHOR ANTI-KOSONG)
    ringkasan_fakta = ""
    # Skenario A: Mengambil teks di antara 'duduk perkara' sampai 'menimbang'
    fakta_match_a = re.search(r'(?:duduk\s+perkara|duduknya\s+perkara)(.*?)(?:menimbang|tentang\s+hukumnya|$)', text)

    # Skenario B (Jika A gagal): Mengambil teks dari 'gugatan pengugat' sampai 'menimbang'
    fakta_match_b = re.search(r'(?:gugatan\s+penggugat|gugatannya)(.*?)(?:menimbang|membaca|$)', text)

    if fakta_match_a and len(fakta_match_a.group(1).strip()) > 100:
        ringkasan_fakta = fakta_match_a.group(1).strip()
    elif fakta_match_b and len(fakta_match_b.group(1).strip()) > 100:
        ringkasan_fakta = fakta_match_b.group(1).strip()
    else:
        # Skenario C (Fallback Terakhir): Mengambil 1500 karakter di tengah dokumen tempat kronologi berada
        mid_point = len(text) // 3
        ringkasan_fakta = text[mid_point:mid_point+1500]

    # Pembersihan kalimat formalitas pengadilan yang sering memicu kesamaan leksikal (TF-IDF)
    procedural_words = [
        r'menerima\s+dan\s+mengutip\s+keadaan-keadaan\s+mengenai\s+duduk\s+perkara.*PN\s+\w+',
        r'salinan\s+resmi\s+putusan\s+pengadilan\s+negeri',
        r'menimbang\s+bahwa\s+tujuan\s+utama',
        r'halaman\s+\d+\s+dari\s+\d+'
    ]
    for pattern in procedural_words:
        ringkasan_fakta = re.sub(pattern, '', ringkasan_fakta, flags=re.IGNORECASE)

    # Ekstraksi Pasal Hukum
    pasal_matches = re.findall(r'(pasal\s+\d+(?:\s+ayat\s+\d+)?\s+(?:kuhperdata|kuhper|undang-undang|uu\s+[^\s]+))', text)
    pasal = ", ".join(list(set(pasal_matches))) if pasal_matches else "1243 KUHPerdata"

    # Ekstraksi Amar Putusan (Solusi)
    amar_match = re.search(r'mengadili\s*:(.*?)(?:demikian|putusan\s+ini|$)', text)
    amar_putusan = amar_match.group(1).strip() if amar_match else "Gugatan dikabulkan untuk sebagian dan menyatakan Tergugat melakukan wanprestasi."

    word_count = len(text.split())


    return {
        "no_perkara": no_perkara,
        "tanggal": tanggal,
        "pihak": pihak,
        "ringkasan_fakta": ringkasan_fakta[:2000].strip(),
        "pasal": pasal,
        "solusi_amar": amar_putusan[:2000].strip(),
        "length": word_count
    }

## Pembuatan Berkas Representasi Kasus

In [3]:


csv_output_path = os.path.join(PROCESSED_DIR, 'cases.csv')
json_output_path = os.path.join(PROCESSED_DIR, 'cases.json')

# Ambil semua file teks hasil Tahap 1
txt_files = [f for f in os.listdir(RAW_DIR) if f.lower().endswith('.txt')]

if len(txt_files) == 0:
    print(f"[ERROR] Folder '{RAW_DIR}' kosong! Pastikan file teks Tahap 1 sudah tergenerasi.")
else:
    extracted_cases = []
    print(f"Memulai pemrosesan {len(txt_files)} dokumen teks ke bentuk terstruktur...\n" + "="*60)

    for filename in sorted(txt_files):
        # Ambil nomor ID kasus dari nama file
        case_id = int(re.search(r'\d+', filename).group())
        txt_path = os.path.join(RAW_DIR, filename)

        with open(txt_path, 'r', encoding='utf-8') as f:
            content = f.read()

        # Jalankan fungsi ekstraktor
        features = extract_case_features(content)

        # Gabungkan data ke dalam kamus
        case_data = {
            "case_id": case_id,
            "no_perkara": features["no_perkara"],
            "tanggal": features["tanggal"],
            "pihak": features["pihak"],
            "ringkasan_fakta": features["ringkasan_fakta"],
            "pasal": features["pasal"],
            "solusi_amar": features["solusi_amar"],
            "length": features["length"],
            "text_full": content
        }
        extracted_cases.append(case_data)
        print(f"Sukses mengekstrak -> {filename} | No. Perkara: {features['no_perkara']}")

    # Ubah list menjadi Pandas DataFrame
    df_cases = pd.DataFrame(extracted_cases)

    # Simpan ke Drive dalam format CSV dan JSON sesuai aturan tugas
    df_cases.to_csv(csv_output_path, index=False, encoding='utf-8')
    df_cases.to_json(json_output_path, orient='records', indent=4, force_ascii=False)

    print("="*60)
    print("TAHAP 2 SELESAI!")
    print(f"File Terstruktur Berhasil Disimpan:\n1. CSV -> {csv_output_path}\n2. JSON -> {json_output_path}")

    # Tampilkan preview 3 data teratas di bawah sel
    print("\nPreview Dataset Terstruktur:")
    display(df_cases[['case_id', 'no_perkara', 'tanggal', 'length']].head(3))

Memulai pemrosesan 30 dokumen teks ke bentuk terstruktur...
Sukses mengekstrak -> case_001.txt | No. Perkara: 117/PDT/2023/PT
Sukses mengekstrak -> case_003.txt | No. Perkara: 172/PDT/2024/PT
Sukses mengekstrak -> case_005.txt | No. Perkara: 28/PDT.G/2020/PN.KRS
Sukses mengekstrak -> case_006.txt | No. Perkara: 18/PDT/2023/PT
Sukses mengekstrak -> case_008.txt | No. Perkara: 265/PDT/2020/PT
Sukses mengekstrak -> case_009.txt | No. Perkara: 276/PDT/2023/PT
Sukses mengekstrak -> case_010.txt | No. Perkara: 296/PDT/2021/PT
Sukses mengekstrak -> case_011.txt | No. Perkara: 312/PDT/2018/PT
Sukses mengekstrak -> case_012.txt | No. Perkara: 312/PDT/2020/PT
Sukses mengekstrak -> case_013.txt | No. Perkara: 333/PDT/2020/PT
Sukses mengekstrak -> case_014.txt | No. Perkara: 353/PDT/2025/PT
Sukses mengekstrak -> case_015.txt | No. Perkara: 379/PDT/2020/PT
Sukses mengekstrak -> case_016.txt | No. Perkara: 385/PDT/2021/PT
Sukses mengekstrak -> case_017.txt | No. Perkara: 38/PDT/2023/PT
Sukses mengek

,case_id,no_perkara,tanggal,length
0,1,117/PDT/2023/PT,30 agustus 2022,2275
1,3,172/PDT/2024/PT,17 februari 2024,1568
2,5,28/PDT.G/2020/PN.KRS,15 januari 2021,2338
